# Calgary 311 Service Requests: SQL Analysis
## Part 2: Answering the business questions

Part 1 built a clean `requests` table. Part 2 uses SQL to answer five business questions:

1. **Demand:** How has request volume changed, when is the busy season, and what drove the 2023 peak?
2. **Workload:** Which services and departments carry the most requests?
3. **Service performance:** How fast are requests resolved, which departments are slowest, and is anything getting slower?
4. **Channels:** How do Calgarians contact 311, and is that changing?
5. **Backlog:** Which requests are still open, for how long, and in which departments?

**New SQL skills in this part:** `LAG()` for year-over-year change, running totals with `SUM() OVER (ORDER BY ...)`, `RANK()`, percentiles, `FILTER` for pivoting, and joining a query to itself through CTEs.

**Every query is saved automatically.** The `run()` helper saves each query as a `.sql` file and its result as a `.csv` file in your Drive project folder. In Part 4, these files become your GitHub repository.

## Step 1: Connect to the database from Part 1

In [ ]:
!pip -q install duckdb

import os, datetime
import duckdb
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/calgary_311_project"
except Exception as e:
    print("Drive not available, using local folder:", e)
    PROJECT_DIR = "calgary_311_project"

DB_PATH = os.path.join(PROJECT_DIR, "calgary_311.duckdb")
SQL_DIR = os.path.join(PROJECT_DIR, "sql")
RESULTS_DIR = os.path.join(PROJECT_DIR, "results")
os.makedirs(SQL_DIR, exist_ok=True); os.makedirs(RESULTS_DIR, exist_ok=True)

con = duckdb.connect(DB_PATH)
pd.set_option("display.max_columns", 50); pd.set_option("display.width", 200)

def q(sql):
    """Run a SQL query and show the result (not saved)."""
    return con.sql(sql).df()

def run(name, sql):
    """Run a SQL query, save it to sql/<name>.sql and its result to results/<name>.csv."""
    with open(os.path.join(SQL_DIR, f"{name}.sql"), "w") as f:
        f.write(sql.strip() + "\n")
    df = con.sql(sql).df()
    df.to_csv(os.path.join(RESULTS_DIR, f"{name}.csv"), index=False)
    return df

print(f"{q('SELECT COUNT(*) AS n FROM requests')['n'][0]:,} requests in the clean table")

## Step 2: Apply the decisions from Part 1

From the Part 1 checks, we decided to:
- **Drop** the 36 rows marked `TO BE DELETED`.
- **Flag duplicates** (`Duplicate (Closed)` and `Duplicate (Open)`). They're reports merged into an earlier request about the same issue, and they're usually closed instantly, which would make resolution times look faster than they are. We exclude them from demand and performance analysis.

A **view** is a saved query that behaves like a table. Every query in this notebook reads from `requests_v` so these rules apply everywhere, consistently.

In [ ]:
con.sql("""
    CREATE OR REPLACE VIEW requests_v AS
    SELECT
        *,
        status IN ('Duplicate (Closed)', 'Duplicate (Open)') AS is_duplicate
    FROM requests
    WHERE status <> 'TO BE DELETED'
""")
q("SELECT COUNT(*) AS rows_in_view, SUM(CAST(is_duplicate AS INT)) AS duplicates FROM requests_v")

### Check: do the status and the closing date agree?
In Part 1, about 46,000 requests had an open status, but only about 43,000 were missing a closing date. This cross-tab shows exactly where they disagree.

In [ ]:
run("00_status_vs_closed_date", """
    SELECT
        status,
        COUNT(*) FILTER (WHERE closed_at IS NULL)     AS no_closed_date,
        COUNT(*) FILTER (WHERE closed_at IS NOT NULL) AS has_closed_date
    FROM requests_v
    GROUP BY status
    ORDER BY status
""")

**How we'll handle it.** For the backlog (Question 5), a request counts as open only if its **status is Open and it has no closing date**, so both sources agree. The small disagreement will be noted in the README as a data-quality caveat.

---
## Question 1: Demand trends

### 1a. Year-over-year change
`LAG(requests) OVER (ORDER BY request_year)` looks at the **previous row's** value, here the previous year's total, so we can calculate the percentage change without a self-join.

In [ ]:
yearly = run("01a_yearly_demand", """
    WITH yearly AS (
        SELECT request_year, COUNT(*) AS requests
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY request_year
    )
    SELECT
        request_year,
        requests,
        LAG(requests) OVER (ORDER BY request_year)                          AS previous_year,
        ROUND(100.0 * (requests - LAG(requests) OVER (ORDER BY request_year))
              / LAG(requests) OVER (ORDER BY request_year), 1)              AS yoy_change_pct
    FROM yearly
    ORDER BY request_year
""")
yearly

### 1b. Seasonality: which months are busiest?
The average number of requests for each calendar month across the five years. Dividing by `COUNT(DISTINCT request_year)` gives an average rather than a five-year total.

In [ ]:
seasonal = run("01b_seasonality", """
    SELECT
        MONTH(requested_at)                                          AS month_num,
        MONTHNAME(requested_at)                                      AS month,
        ROUND(COUNT(*) / COUNT(DISTINCT request_year), 0)            AS avg_requests_per_year
    FROM requests_v
    WHERE NOT is_duplicate
    GROUP BY month_num, month
    ORDER BY month_num
""")
ax = seasonal.plot(x="month", y="avg_requests_per_year", kind="bar", legend=False, figsize=(9, 3.5), color="#4c72b0")
ax.set_title("Average 311 requests by month (2021–2025)"); ax.set_xlabel(""); ax.set_ylabel("Requests")
plt.tight_layout(); plt.show()
seasonal

### 1c. What drove the 2023 peak?
We **pivot** the data so each service type becomes a row with one column per year. `COUNT(*) FILTER (WHERE ...)` counts only the rows that match the condition. It works in DuckDB and PostgreSQL. In SQL Server you'd write `SUM(CASE WHEN request_year = 2023 THEN 1 ELSE 0 END)`, which does the same thing.

In [ ]:
drivers = run("01c_drivers_of_2023_peak", """
    WITH by_service AS (
        SELECT
            service_name,
            COUNT(*) FILTER (WHERE request_year = 2022) AS y2022,
            COUNT(*) FILTER (WHERE request_year = 2023) AS y2023,
            COUNT(*) FILTER (WHERE request_year = 2024) AS y2024,
            COUNT(*) FILTER (WHERE request_year = 2025) AS y2025
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY service_name
    )
    SELECT *,
           y2023 - y2022 AS change_2022_to_2023,
           y2025 - y2023 AS change_2023_to_2025
    FROM by_service
    ORDER BY change_2022_to_2023 DESC
    LIMIT 10
""")
drivers

In [ ]:
# The other side: which services declined most after the peak?
q("""
    WITH by_service AS (
        SELECT
            service_name,
            COUNT(*) FILTER (WHERE request_year = 2023) AS y2023,
            COUNT(*) FILTER (WHERE request_year = 2025) AS y2025
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY service_name
    )
    SELECT *, y2025 - y2023 AS change_2023_to_2025
    FROM by_service
    ORDER BY change_2023_to_2025 ASC
    LIMIT 10
""")

---
## Question 2: Workload

### 2a. Pareto analysis: do a few service types make up most of the volume?
A **running total** (`SUM(requests) OVER (ORDER BY requests DESC ...)`) adds up volume from the biggest service type down. Dividing it by the grand total (`SUM(requests) OVER ()`) shows what share the top N services cover. This is the classic "80/20" check.

In [ ]:
pareto = run("02a_service_pareto", """
    WITH services AS (
        SELECT service_name, agency, COUNT(*) AS requests
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY service_name, agency
    ),
    ranked AS (
        SELECT *,
               RANK() OVER (ORDER BY requests DESC)                                   AS rnk,
               SUM(requests) OVER (ORDER BY requests DESC
                                   ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)  AS running_total,
               SUM(requests) OVER ()                                                  AS grand_total
        FROM services
    )
    SELECT rnk, service_name, agency, requests,
           ROUND(100.0 * requests / grand_total, 2)      AS pct_of_all,
           ROUND(100.0 * running_total / grand_total, 1) AS cumulative_pct
    FROM ranked
    ORDER BY rnk
    LIMIT 15
""")
pareto

In [ ]:
run("02b_pareto_summary", """
    WITH services AS (
        SELECT service_name, COUNT(*) AS requests
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY service_name
    ),
    ranked AS (
        SELECT ROW_NUMBER() OVER (ORDER BY requests DESC) AS rnk,
               100.0 * SUM(requests) OVER (ORDER BY requests DESC
                                           ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
                     / SUM(requests) OVER () AS cumulative_pct
        FROM services
    )
    SELECT
        COUNT(*)                                   AS total_service_types,
        MIN(rnk) FILTER (WHERE cumulative_pct >= 50) AS types_covering_50_pct,
        MIN(rnk) FILTER (WHERE cumulative_pct >= 80) AS types_covering_80_pct
    FROM ranked
""")

### 2c. Workload by department

In [ ]:
run("02c_agency_workload", """
    SELECT
        agency,
        COUNT(*)                                            AS requests,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)  AS pct_of_all,
        COUNT(DISTINCT service_name)                        AS service_types
    FROM requests_v
    WHERE NOT is_duplicate
    GROUP BY agency
    ORDER BY requests DESC
    LIMIT 15
""")

---
## Question 3: Service performance

### 3a. How fast does each department resolve requests?
The **median** is the typical case. The **90th percentile (P90)** is how long the slowest 10% take, which is where service problems show up. `PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY ...)` is standard SQL that also works in PostgreSQL and SQL Server.

`HAVING COUNT(*) >= 5000` keeps only departments with enough requests for the numbers to be reliable.

In [ ]:
perf = run("03a_agency_resolution_time", """
    SELECT
        agency,
        COUNT(*)                                                         AS closed_requests,
        MEDIAN(resolution_days)                                          AS median_days,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY resolution_days)     AS p90_days,
        ROUND(100.0 * AVG(CASE WHEN resolution_days <= 7 THEN 1 ELSE 0 END), 1) AS pct_closed_within_7_days
    FROM requests_v
    WHERE NOT is_duplicate
      AND resolution_days IS NOT NULL
    GROUP BY agency
    HAVING COUNT(*) >= 5000
    ORDER BY p90_days DESC
""")
ax = perf.sort_values("p90_days").plot(x="agency", y=["median_days", "p90_days"], kind="barh", figsize=(9, 5))
ax.set_title("Days to resolve a request, by department"); ax.set_xlabel("Days"); ax.set_ylabel("")
plt.tight_layout(); plt.show()
perf

### 3b. Slowest individual service types (at least 1,000 requests)

In [ ]:
run("03b_slowest_services", """
    SELECT
        service_name,
        agency,
        COUNT(*)                                                      AS closed_requests,
        MEDIAN(resolution_days)                                       AS median_days,
        PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY resolution_days)  AS p90_days
    FROM requests_v
    WHERE NOT is_duplicate
      AND resolution_days IS NOT NULL
    GROUP BY service_name, agency
    HAVING COUNT(*) >= 1000
    ORDER BY p90_days DESC
    LIMIT 15
""")

### 3c. Is any department getting slower?
We calculate P90 per department per year, then pivot to compare 2021 with 2025.

**A caveat worth knowing (and mentioning in interviews):** requests that are still open have no resolution time yet, so they're left out. Recent years have had less time to close their slow cases, which can make recent years look slightly *faster* than they really are. This is called **right-censoring**. Since about 98% of requests are closed, the effect here is small, but a careful analyst names it.

In [ ]:
run("03c_agency_trend", """
    WITH yearly AS (
        SELECT agency, request_year,
               PERCENTILE_CONT(0.9) WITHIN GROUP (ORDER BY resolution_days) AS p90_days
        FROM requests_v
        WHERE NOT is_duplicate
          AND resolution_days IS NOT NULL
        GROUP BY agency, request_year
        HAVING COUNT(*) >= 1000
    )
    SELECT
        agency,
        MAX(p90_days) FILTER (WHERE request_year = 2021) AS p90_2021,
        MAX(p90_days) FILTER (WHERE request_year = 2023) AS p90_2023,
        MAX(p90_days) FILTER (WHERE request_year = 2025) AS p90_2025,
        MAX(p90_days) FILTER (WHERE request_year = 2025)
          - MAX(p90_days) FILTER (WHERE request_year = 2021) AS change_2021_to_2025
    FROM yearly
    GROUP BY agency
    HAVING COUNT(*) = 5              -- only departments with enough data in all five years
    ORDER BY change_2021_to_2025 DESC
""")

---
## Question 4: Channels

### 4a. How has the channel mix changed each year?

In [ ]:
channels = run("04a_channel_mix_by_year", """
    SELECT
        request_year,
        ROUND(100.0 * COUNT(*) FILTER (WHERE channel = 'Phone') / COUNT(*), 1) AS phone_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE channel = 'App')   / COUNT(*), 1) AS app_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE channel = 'Web')   / COUNT(*), 1) AS web_pct,
        ROUND(100.0 * COUNT(*) FILTER (WHERE channel = 'Other') / COUNT(*), 1) AS other_pct
    FROM requests_v
    WHERE NOT is_duplicate
    GROUP BY request_year
    ORDER BY request_year
""")
ax = channels.plot(x="request_year", y=["phone_pct", "app_pct", "web_pct", "other_pct"], marker="o", figsize=(8, 3.5))
ax.set_title("Share of 311 requests by channel"); ax.set_xlabel(""); ax.set_ylabel("% of requests")
plt.tight_layout(); plt.show()
channels

### 4b. What comes through the "Other" channel?
"Other" is the largest channel, but the dataset doesn't define it. Looking at which services use it can hint at what it represents. We report what we find and **don't guess** beyond the data.

In [ ]:
run("04b_other_channel_services", """
    SELECT
        service_name,
        agency,
        COUNT(*)                                                     AS requests,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)           AS pct_of_other_channel
    FROM requests_v
    WHERE NOT is_duplicate
      AND channel = 'Other'
    GROUP BY service_name, agency
    ORDER BY requests DESC
    LIMIT 10
""")

---
## Question 5: Backlog

### 5a. How old are the open requests?
A request counts as open when its status is `Open` **and** it has no closing date. We sort open requests into age buckets with `CASE WHEN`, measured up to the date you run this notebook.

In [ ]:
AS_OF = datetime.date.today().isoformat()
print("Ages measured as of", AS_OF)

backlog_age = run("05a_backlog_aging", f"""
    WITH open_requests AS (
        SELECT *, DATE_DIFF('day', CAST(requested_at AS DATE), DATE '{AS_OF}') AS age_days
        FROM requests_v
        WHERE status = 'Open' AND closed_at IS NULL
    )
    SELECT
        CASE
            WHEN age_days < 365       THEN '1. Under 1 year'
            WHEN age_days < 2 * 365   THEN '2. 1-2 years'
            WHEN age_days < 3 * 365   THEN '3. 2-3 years'
            ELSE                           '4. Over 3 years'
        END                                                  AS age_bucket,
        COUNT(*)                                             AS open_requests,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)   AS pct_of_backlog
    FROM open_requests
    GROUP BY age_bucket
    ORDER BY age_bucket
""")
backlog_age

### 5b. Which departments and services hold the backlog?

In [ ]:
run("05b_backlog_by_agency", f"""
    WITH open_requests AS (
        SELECT *, DATE_DIFF('day', CAST(requested_at AS DATE), DATE '{AS_OF}') AS age_days
        FROM requests_v
        WHERE status = 'Open' AND closed_at IS NULL
    ),
    agency_totals AS (                        -- all non-duplicate requests per department, for context
        SELECT agency, COUNT(*) AS total_requests
        FROM requests_v
        WHERE NOT is_duplicate
        GROUP BY agency
    )
    SELECT
        o.agency,
        COUNT(*)                                              AS open_requests,
        ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1)    AS pct_of_backlog,
        ROUND(100.0 * COUNT(*) / MAX(t.total_requests), 2)    AS pct_of_agency_requests_still_open,
        MEDIAN(o.age_days)                                    AS median_age_days
    FROM open_requests o
    JOIN agency_totals t ON t.agency = o.agency               -- a JOIN combines the two results by department
    GROUP BY o.agency
    ORDER BY open_requests DESC
    LIMIT 10
""")

In [ ]:
run("05c_backlog_by_service", """
    SELECT service_name, agency, COUNT(*) AS open_requests
    FROM requests_v
    WHERE status = 'Open' AND closed_at IS NULL
    GROUP BY service_name, agency
    ORDER BY open_requests DESC
    LIMIT 10
""")

---
## Preview of Part 3: communities

Raw counts by community mostly reflect **how many people live there**. That's why Part 3 joins population data to compare requests **per 1,000 residents**. Here's the raw view for reference, plus the check on requests with no community.

In [ ]:
run("06a_top_communities_raw", """
    SELECT community, COUNT(*) AS requests
    FROM requests_v
    WHERE NOT is_duplicate AND community IS NOT NULL
    GROUP BY community
    ORDER BY requests DESC
    LIMIT 10
""")

In [ ]:
run("06b_requests_without_community", """
    SELECT service_name, COUNT(*) AS requests,
           ROUND(100.0 * COUNT(*) / SUM(COUNT(*)) OVER (), 1) AS pct_of_no_community
    FROM requests_v
    WHERE NOT is_duplicate AND community IS NULL
    GROUP BY service_name
    ORDER BY requests DESC
    LIMIT 10
""")

---
## Checkpoint

Every query above is now saved in your Drive under `calgary_311_project/sql/` and its result under `calgary_311_project/results/`.

**Send me the outputs of these cells:** 00, 1a, 1c (both tables), 2a, 2b, 2c, 3a, 3c, 4a, 4b, 5a, 5b. Screenshots are fine. We'll turn them into findings together, and then move on to Part 3.

In [ ]:
con.close()
print("Done. Files saved in", PROJECT_DIR)